# 1. 라이브러리 호출

In [98]:
from pathlib import Path
import ast
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 한글 폰트 설정
# Windows에서는 Malgun Gothic, macOS에서는 AppleGothic, 그 외 환경에서는 NanumGothic을 사용합니다.
import platform

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (12, 6)

# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)

# 2. 파일 경로 설정 및 데이터 읽기

In [99]:
# 프로젝트 루트 직접 지정
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정합니다.
ROOT = Path(r"C:\Users\joon5\Documents\github\Anti-Churn-Committee")

# 원본 데이터가 들어있는 폴더
DATA_RAW_DIR = ROOT / "data" / "raw"

# 전처리 데이터 저장 폴더
DATA_PROCESSED_DIR = ROOT / "data" / "processed"

# 실제 분석에 사용할 파일명
USER_PROFILE_FILE = "01_User_Profile.csv"
EVENT_LOG_FILE = "02_Event_Log.csv"

# 전체 경로 생성
USER_PROFILE_PATH = DATA_RAW_DIR / USER_PROFILE_FILE
EVENT_LOG_PATH = DATA_RAW_DIR / EVENT_LOG_FILE

print("호출 폴더")
print("ROOT:", ROOT)
print("DATA_RAW_DIR:", DATA_RAW_DIR)
print("DATA_PROCESSED_DIR:", DATA_PROCESSED_DIR)
print("유저 프로필 파일:", USER_PROFILE_PATH)
print("이벤트 로그 파일:", EVENT_LOG_PATH)

호출 폴더
ROOT: C:\Users\joon5\Documents\github\Anti-Churn-Committee
DATA_RAW_DIR: C:\Users\joon5\Documents\github\Anti-Churn-Committee\data\raw
DATA_PROCESSED_DIR: C:\Users\joon5\Documents\github\Anti-Churn-Committee\data\processed
유저 프로필 파일: C:\Users\joon5\Documents\github\Anti-Churn-Committee\data\raw\01_User_Profile.csv
이벤트 로그 파일: C:\Users\joon5\Documents\github\Anti-Churn-Committee\data\raw\02_Event_Log.csv


In [100]:
df_raw01 = pd.read_csv(USER_PROFILE_PATH)
df_raw02 = pd.read_csv(EVENT_LOG_PATH)

print("df_raw01 shape :", df_raw01.shape)
print("df_raw02 shape :", df_raw02.shape)

df_raw01 shape : (12500, 6)
df_raw02 shape : (1757262, 5)


In [101]:
user_profile = df_raw01.copy()
event_log = df_raw02.copy()

In [102]:
user_profile.head()

,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자
0,U0000001,2025-01-25,오가닉,iOS,True,NaN
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24
2,U0000003,2025-05-14,오가닉,iOS,False,NaN
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaN
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaN


In [103]:
event_log.head()

,User_ID,Event_Time,Event_Type,Session_ID,알림_유형
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,NaN
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,NaN
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,NaN
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,NaN
4,U0000001,2025-01-25 20:30:00,알림수신,NaN,광고성


# 3. 데이터 검수

## 3-1 범용 함수

In [104]:
def check_basic_info(df, df_name, exclude_cols=None):
    """
    행/열 수, 완전 중복 행, 컬럼별 타입/결측/고유값을 한 번에 확인
    """

    print(f"\n{'='*80}")
    print(f"{df_name}의 기본 정보 / 타입 / 결측치 확인")
    print(f"{'='*80}\n")

    df_copied = df.copy()

    # 제외할 컬럼 반영
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # dict, list, set 같은 해시 불가능 값이 들어있는 컬럼은 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)

    # 잘못된 데이터프레임 종료
    row_count = len(df_copied)
    if row_count == 0:
        print("데이터프레임이 비어 있습니다.")
        return

    # 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })

    # 컬럼별 요약
    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / row_count * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / row_count * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True),
        '고유값 비율(%)': (df_copied.nunique(dropna=True) / row_count * 100).round(2)
    }).sort_values(
        by=['결측치 개수', '고유값 개수'],
        ascending=[False, False]
    )

    print(f"{df_name} [전체 요약]")
    display(overview_df)

    print(f"{df_name} [컬럼별 요약]")
    display(summary_df)

    print(f"{df_name} [상위 5행]")
    display(df_copied.head())

In [105]:
def check_id_duplicates(df, col_name, df_name, top_n=10):
    """
    기준 키로 사용할 컬럼의 중복 여부를 확인한다.
    단일 컬럼 또는 여러 컬럼 조합을 기준으로 확인할 수 있다.
    """

    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 값 중복 확인")
    print(f"{'='*80}")

    # 원본 데이터 훼손 방지
    df_copied = df.copy()

    # 단일 컬럼 기준 확인
    if isinstance(col_name, str):
        if col_name not in df_copied.columns:
            print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
            return

        key_series = df_copied[col_name].fillna("결측").astype(str)
        key_label = col_name

    # 여러 컬럼 조합 기준 확인
    else:
        missing_cols = [col for col in col_name if col not in df_copied.columns]
        if missing_cols:
            print(f"존재하지 않는 컬럼이 있습니다: {missing_cols}")
            return

        # 여러 컬럼을 하나의 문자열 키로 결합
        # 결측치는 먼저 "결측"으로 채운 뒤 문자열로 변환한다.
        key_series = (
            df_copied[col_name]
            .fillna("결측")
            .astype(str)
            .agg(" | ".join, axis=1)
        )

        key_label = " + ".join(col_name)

    # 중복 개수 계산
    duplicate_count = key_series.duplicated().sum()

    print("전체 행 수:", len(df_copied))
    print(f"{key_label} 고유 개수:", key_series.nunique(dropna=False))
    print(f"중복 {key_label} 개수:", duplicate_count)

    # 중복 값이 있는 경우 상위 값 확인
    if duplicate_count > 0:
        print("\n[중복 상위 값]")
        dup_summary = key_series.value_counts(dropna=False).reset_index()
        dup_summary.columns = [key_label, "등장 횟수"]
        display(dup_summary[dup_summary["등장 횟수"] > 1].head(top_n))
    else:
        print("중복 값이 없습니다.")
# 이 코드를 진짜 몇번이고 개량하는지 모르겠네

In [106]:
def check_duplicate_except_cols(df, exclude_cols, df_name, top_n=10):
    """
    특정 컬럼을 제외했을 때, 나머지 값이 완전히 동일한 행이 있는지 확인한다.
    """

    print(f"\n{'='*80}")
    print(f"{df_name}의 특정 컬럼 제외 후 중복 확인")
    print(f"{'='*80}")

    # 원본 데이터 훼손 방지
    df_copied = df.copy()

    # 제외할 컬럼 반영
    check_df = df_copied.drop(columns=exclude_cols, errors='ignore')

    # 잘못된 데이터프레임 종료
    if check_df.shape[1] == 0:
        print("검사할 컬럼이 없습니다.")
        return

    # 특정 컬럼 제외 후 중복 개수 계산
    duplicate_count = check_df.duplicated().sum()

    print("전체 행 수:", len(df_copied))
    print("제외 컬럼:", exclude_cols)
    print("검사 대상 컬럼 수:", check_df.shape[1])
    print("특정 컬럼 제외 후 중복 행 개수:", duplicate_count)

    # 중복 행이 있는 경우 샘플 확인
    if duplicate_count > 0:
        print("\n[중복 행 샘플]")
        dup_mask = check_df.duplicated(keep=False)
        display(df_copied[dup_mask].head(top_n))
    else:
        print("특정 컬럼을 제외해도 중복 행이 없습니다.")
    
# 이 코드를 진짜 몇번이고 개량하는지 모르겠네

In [107]:
def check_datetime_columns(df, date_cols, df_name):
    """
    날짜 관련 컬럼이 정상적으로 datetime 타입으로 변환되는지 확인한다.
    """

    print(f"\n{'='*80}")
    print(f"{df_name}의 날짜 변환 상태 확인")
    print(f"{'='*80}")

    # 원본 데이터 훼손 방지
    df_copied = df.copy()

    # 날짜 변환 결과 저장
    result_list = []

    for col in date_cols:
        # 컬럼 존재 여부 확인
        if col not in df_copied.columns:
            result_list.append({
                '컬럼명': col,
                '상태': '컬럼 없음',
                '원본 결측치 개수': None,
                '변환 실패 개수': None,
                '최소 날짜': None,
                '최대 날짜': None
            })
            continue

        # 날짜 변환
        converted = pd.to_datetime(df_copied[col], errors='coerce')

        # 원본 결측과 변환 실패 구분
        original_missing = df_copied[col].isnull().sum()
        converted_missing = converted.isnull().sum()
        parse_fail_count = converted_missing - original_missing

        result_list.append({
            '컬럼명': col,
            '상태': '확인 완료',
            '원본 결측치 개수': original_missing,
            '변환 실패 개수': parse_fail_count,
            '최소 날짜': converted.min(),
            '최대 날짜': converted.max()
        })

    # 결과 요약
    result_df = pd.DataFrame(result_list)
    display(result_df)

In [108]:
# 1. 기술통계
def check_statistics_summary(df, df_name, exclude_cols=None):
    """
    수치형 컬럼의 기술통계량과 왜도를 확인한다.
    """
    
    print(f"\n{'='*80}")
    print(f"{df_name} 변수 기술통계량")
    print(f"{'='*80}\n")
    
    # exclude_cols에 속한 컬럼 제외한 데이터프레임 생성
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')
    
    # 수치형 컬럼만 선택
    numeric_df = df_copied.select_dtypes(include='number')
    
    if numeric_df.shape[1] == 0:
        print("수치형 컬럼이 없습니다.")
        return
    
    # 기술 통계량
    stats_df = numeric_df.describe().T
    stats_df['Skewness'] = numeric_df.skew()
    
    # 컬럼명 한글로 변경
    stats_df.rename(columns={
        'count': '개수',
        'mean': '평균',
        'std': '표준편차',
        'min': '최솟값',
        '25%': 'Q1의 경계값',
        '50%': '중앙값',
        '75%': 'Q3의 경계값',
        'max': '최댓값',
        'Skewness': '왜도'
    }, inplace=True)
    
    display(stats_df)

In [109]:
# 2. IQR 기준 이상치 후보 확인
def check_outliers_iqr(df, df_name, exclude_cols=None):
    """
    수치형 컬럼에 대해 IQR 기준 이상치 후보를 확인한다.
    """

    print(f"\n{'='*80}")
    print(f"{df_name} IQR 기준 이상치 후보 확인")
    print(f"{'='*80}\n")

    # 원본 데이터 훼손 방지
    df_copied = df.copy()

    # 제외할 컬럼 반영
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # 수치형 컬럼만 선택
    numeric_df = df_copied.select_dtypes(include='number')

    # 잘못된 데이터프레임 종료
    if numeric_df.shape[1] == 0:
        print("수치형 컬럼이 없습니다.")
        return

    # IQR 기준 이상치 후보 계산
    result_list = []

    for col in numeric_df.columns:
        q1 = numeric_df[col].quantile(0.25)
        q3 = numeric_df[col].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_count = (
            (numeric_df[col] < lower_bound) |
            (numeric_df[col] > upper_bound)
        ).sum()

        result_list.append({
            '컬럼명': col,
            'Q1': q1,
            'Q3': q3,
            'IQR': iqr,
            '하한값': lower_bound,
            '상한값': upper_bound,
            '이상치 후보 개수': outlier_count,
            '이상치 후보 비율(%)': round(outlier_count / len(numeric_df) * 100, 2)
        })

    # 결과 요약
    result_df = pd.DataFrame(result_list)
    display(result_df)

In [110]:
def check_category_values(df, df_name, col_name, top_n=10):
    """
    범주형 컬럼에 어떤 값이 있는지 개수와 비율로 확인한다.
    """

    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 값 확인")
    print(f"{'='*80}")

    # 원본 데이터 훼손 방지
    df_copied = df.copy()

    # 컬럼 존재 여부 확인
    if col_name not in df_copied.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    # 잘못된 데이터프레임 종료
    row_count = len(df_copied)
    if row_count == 0:
        print("데이터프레임이 비어 있습니다.")
        return

    # 범주별 개수 계산, 결측치 포함
    summary_df = df_copied[col_name].value_counts(dropna=False).reset_index()

    # 결과 컬럼명 정리
    summary_df.columns = [col_name, '개수']

    # 범주별 비율 계산
    summary_df['비율(%)'] = (summary_df['개수'] / row_count * 100).round(2)

    print("전체 행 수:", row_count)
    print(f"{col_name} 고유값 개수(결측 포함):", df_copied[col_name].nunique(dropna=False))
    print()

    print("[범주별 요약]")
    display(summary_df.head(top_n))

In [111]:
def check_user_id_relation(df_profile, df_event):
    """
    User_Profile과 Event_Log의 User_ID 연결 상태를 확인한다.
    """

    print(f"\n{'='*80}")
    print("User_Profile - Event_Log User_ID 연결 확인")
    print(f"{'='*80}")

    # 원본 데이터 훼손 방지
    profile_copied = df_profile.copy()
    event_copied = df_event.copy()

    # 컬럼 존재 여부 확인
    if 'User_ID' not in profile_copied.columns:
        print("User_Profile에 'User_ID' 컬럼이 존재하지 않습니다.")
        return

    if 'User_ID' not in event_copied.columns:
        print("Event_Log에 'User_ID' 컬럼이 존재하지 않습니다.")
        return

    # User_ID 집합 생성
    profile_users = set(profile_copied['User_ID'].dropna())
    event_users = set(event_copied['User_ID'].dropna())

    # 데이터 간 연결 상태 계산
    both_users = profile_users & event_users
    only_profile_users = profile_users - event_users
    only_event_users = event_users - profile_users

    # 연결 상태 요약
    relation_df = pd.DataFrame({
        '항목': [
            '프로필 User_ID 고유 수',
            '이벤트 User_ID 고유 수',
            '양쪽 모두 존재하는 User_ID 수',
            '프로필에만 있는 User_ID 수',
            '이벤트에만 있는 User_ID 수'
        ],
        '값': [
            len(profile_users),
            len(event_users),
            len(both_users),
            len(only_profile_users),
            len(only_event_users)
        ]
    })

    display(relation_df)

    # 프로필에만 존재하는 User_ID 샘플 확인
    if len(only_profile_users) > 0:
        print("\n[프로필에는 있지만 이벤트 로그에는 없는 User_ID 샘플]")
        display(pd.DataFrame({'User_ID': list(only_profile_users)[:10]}))

    # 이벤트 로그에만 존재하는 User_ID 샘플 확인
    if len(only_event_users) > 0:
        print("\n[이벤트 로그에는 있지만 프로필에는 없는 User_ID 샘플]")
        display(pd.DataFrame({'User_ID': list(only_event_users)[:10]}))

## 범용 검수 작업

In [112]:
# 1. 기본 구조 및 결측값 확인
check_basic_info(user_profile, "User_Profile")
check_basic_info(event_log, "Event_Log")


User_Profile의 기본 정보 / 타입 / 결측치 확인

User_Profile [전체 요약]


,항목,값
0,행 개수,12500
1,열 개수,6
2,중복 행 개수,0


User_Profile [컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수,고유값 비율(%)
알림수신동의_변경일자,str,1976,15.81,10524,84.19,148,1.18
가입경로,str,12363,98.90,137,1.10,2,0.02
기기,str,12379,99.03,121,0.97,2,0.02
알림수신동의여부,object,12384,99.07,116,0.93,2,0.02
User_ID,str,12500,100.00,0,0.00,12500,100.00
가입일자,str,12500,100.00,0,0.00,146,1.17


User_Profile [상위 5행]


...

,항목,값
0,행 개수,1757262
1,열 개수,5
2,중복 행 개수,0


Event_Log [컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수,고유값 비율(%)
알림_유형,str,218882,12.46,1538380,87.54,3,0.00
Session_ID,str,1515760,86.26,241502,13.74,736281,41.90
Event_Type,str,1730806,98.49,26456,1.51,10,0.00
Event_Time,str,1757262,100.00,0,0.00,1495136,85.08
User_ID,str,1757262,100.00,0,0.00,12453,0.71


Event_Log [상위 5행]


,User_ID,Event_Time,Event_Type,Session_ID,알림_유형
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,NaN
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,NaN
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,NaN
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,NaN
4,U0000001,2025-01-25 20:30:00,알림수신,NaN,광고성


In [113]:
# 2. 키 중복 확인
check_id_duplicates(user_profile, "User_ID", "User_Profile")
check_id_duplicates(
    event_log,
    ["User_ID", "Event_Time", "Event_Type", "Session_ID", "알림_유형"],
    "Event_Log"
)


User_Profile의 User_ID 값 중복 확인
전체 행 수: 12500
User_ID 고유 개수: 12500
중복 User_ID 개수: 0
중복 값이 없습니다.

Event_Log의 ['User_ID', 'Event_Time', 'Event_Type', 'Session_ID', '알림_유형'] 값 중복 확인
전체 행 수: 1757262
User_ID + Event_Time + Event_Type + Session_ID + 알림_유형 고유 개수: 1757262
중복 User_ID + Event_Time + Event_Type + Session_ID + 알림_유형 개수: 0
중복 값이 없습니다.


In [114]:
# 3. ID 제외 후 값 중복 확인

# User_ID만 제외하고 나머지 값이 모두 같은 경우
check_duplicate_except_cols(
    user_profile,
    ["User_ID"],
    "User_Profile"
)

# User_ID, Session_ID를 제외하고 나머지 로그 값이 같은 경우
check_duplicate_except_cols(
    event_log,
    ["User_ID", "Session_ID"],
    "Event_Log"
)


User_Profile의 특정 컬럼 제외 후 중복 확인
전체 행 수: 12500
제외 컬럼: ['User_ID']
검사 대상 컬럼 수: 5
특정 컬럼 제외 후 중복 행 개수: 9121

[중복 행 샘플]


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자
0,U0000001,2025-01-25,오가닉,iOS,True,NaN
2,U0000003,2025-05-14,오가닉,iOS,False,NaN
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaN
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaN
5,U0000006,2025-02-07,퍼포먼스광고,Android,True,NaN
7,U0000008,2025-02-27,오가닉,iOS,False,NaN
8,U0000009,2025-05-20,퍼포먼스광고,iOS,False,NaN
9,U0000010,2025-04-21,오가닉,iOS,False,NaN
10,U0000011,2025-02-24,오가닉,iOS,True,NaN
11,U0000012,2025-04-21,오가닉,iOS,True,NaN



Event_Log의 특정 컬럼 제외 후 중복 확인
전체 행 수: 1757262
제외 컬럼: ['User_ID', 'Session_ID']
검사 대상 컬럼 수: 3
특정 컬럼 제외 후 중복 행 개수: 87794

[중복 행 샘플]


,User_ID,Event_Time,Event_Type,Session_ID,알림_유형
9,U0000001,2025-01-26 15:18:00,알림수신,NaN,리마인드
13,U0000001,2025-01-26 20:11:12,앱실행,0504a5087b,NaN
15,U0000001,2025-01-26 22:58:09,앱실행,06566985d2,NaN
48,U0000001,2025-01-29 18:23:00,알림수신,NaN,리마인드
111,U0000001,2025-02-04 19:43:00,알림수신,NaN,챌린지_알림
112,U0000001,2025-02-05 09:47:42,앱실행,dbddce03dc,NaN
113,U0000001,2025-02-05 16:14:00,알림수신,NaN,리마인드
124,U0000001,2025-02-06 22:12:29,앱실행,d0e98dba23,NaN
151,U0000001,2025-02-12 09:20:00,알림수신,NaN,리마인드
159,U0000001,2025-02-13 10:37:00,알림수신,NaN,리마인드


In [115]:
# 4. 날짜 컬럼 확인
check_datetime_columns(
    user_profile,
    ["가입일자", "알림수신동의_변경일자"],
    "User_Profile"
)
check_datetime_columns(
    event_log,
    ["Event_Time"],
    "Event_Log"
)


User_Profile의 날짜 변환 상태 확인


,컬럼명,상태,원본 결측치 개수,변환 실패 개수,최소 날짜,최대 날짜
0,가입일자,확인 완료,0,0,2025-01-01,2025-05-26
1,알림수신동의_변경일자,확인 완료,10524,0,2025-01-21,2025-06-30



Event_Log의 날짜 변환 상태 확인


,컬럼명,상태,원본 결측치 개수,변환 실패 개수,최소 날짜,최대 날짜
0,Event_Time,확인 완료,0,0,2025-01-01 07:00:07,2025-06-30 22:59:51


In [116]:
# 5. 기술통계

In [117]:
# 사용자별 이벤트 수
user_event_count = (
    event_log
    .groupby("User_ID")
    .size()
    .reset_index(name="event_count")
)

check_statistics_summary(user_event_count, "사용자별 이벤트 수", exclude_cols=["User_ID"])
check_outliers_iqr(user_event_count, "사용자별 이벤트 수", exclude_cols=["User_ID"])


사용자별 이벤트 수 변수 기술통계량



,개수,평균,표준편차,최솟값,Q1의 경계값,중앙값,Q3의 경계값,최댓값,왜도
event_count,12453.0,141.111539,172.167588,1.0,20.0,51.0,219.0,697.0,1.332713



사용자별 이벤트 수 IQR 기준 이상치 후보 확인



,컬럼명,Q1,Q3,IQR,하한값,상한값,이상치 후보 개수,이상치 후보 비율(%)
0,event_count,20.0,219.0,199.0,-278.5,517.5,681,5.47


In [118]:
# 세션별 이벤트 수
session_event_count = (
    event_log
    .dropna(subset=["Session_ID"])
    .groupby("Session_ID")
    .size()
    .reset_index(name="event_count")
)

check_statistics_summary(session_event_count, "세션별 이벤트 수", exclude_cols=["Session_ID"])
check_outliers_iqr(session_event_count, "세션별 이벤트 수", exclude_cols=["Session_ID"])


세션별 이벤트 수 변수 기술통계량



,개수,평균,표준편차,최솟값,Q1의 경계값,중앙값,Q3의 경계값,최댓값,왜도
event_count,736281.0,2.058671,1.023967,1.0,1.0,2.0,3.0,8.0,0.940113



세션별 이벤트 수 IQR 기준 이상치 후보 확인



,컬럼명,Q1,Q3,IQR,하한값,상한값,이상치 후보 개수,이상치 후보 비율(%)
0,event_count,1.0,3.0,2.0,-2.0,6.0,183,0.02


In [119]:
# 일별 이벤트 수
event_log_date = event_log.copy()
event_log_date["Event_Time"] = pd.to_datetime(event_log_date["Event_Time"], errors="coerce")
event_log_date["Event_Date"] = event_log_date["Event_Time"].dt.date

daily_event_count = (
    event_log_date
    .groupby("Event_Date")
    .size()
    .reset_index(name="event_count")
)

check_statistics_summary(daily_event_count, "일별 이벤트 수", exclude_cols=["Event_Date"])
check_outliers_iqr(daily_event_count, "일별 이벤트 수", exclude_cols=["Event_Date"])


일별 이벤트 수 변수 기술통계량



,개수,평균,표준편차,최솟값,Q1의 경계값,중앙값,Q3의 경계값,최댓값,왜도
event_count,181.0,9708.629834,4497.758932,665.0,5050.0,11358.0,13846.0,16495.0,-0.314043



일별 이벤트 수 IQR 기준 이상치 후보 확인



,컬럼명,Q1,Q3,IQR,하한값,상한값,이상치 후보 개수,이상치 후보 비율(%)
0,event_count,5050.0,13846.0,8796.0,-8144.0,27040.0,0,0.0


In [120]:
# 7. 범주형 값 분포 확인
print("user_profile")
check_category_values(user_profile, "User_Profile", "가입경로") 
check_category_values(user_profile, "User_Profile", "기기") 
check_category_values(user_profile, "User_Profile", "알림수신동의여부")

print("\n event_log")

check_category_values(event_log, "Event_Log", "Event_Type") 
check_category_values(event_log, "Event_Log", "알림_유형")

user_profile

User_Profile의 가입경로 값 확인
전체 행 수: 12500
가입경로 고유값 개수(결측 포함): 3

[범주별 요약]


,가입경로,개수,비율(%)
0,퍼포먼스광고,6852,54.82
1,오가닉,5511,44.09
2,NaN,137,1.10



User_Profile의 기기 값 확인
전체 행 수: 12500
기기 고유값 개수(결측 포함): 3

[범주별 요약]


,기기,개수,비율(%)
0,iOS,7175,57.40
1,Android,5204,41.63
2,NaN,121,0.97



User_Profile의 알림수신동의여부 값 확인
전체 행 수: 12500
알림수신동의여부 고유값 개수(결측 포함): 3

[범주별 요약]


,알림수신동의여부,개수,비율(%)
0,True,7984,63.87
1,False,4400,35.20
2,NaN,116,0.93



 event_log

Event_Log의 Event_Type 값 확인
전체 행 수: 1757262
Event_Type 고유값 개수(결측 포함): 11

[범주별 요약]


,Event_Type,개수,비율(%)
0,앱실행,728657,41.47
1,수면기록,242978,13.83
2,알림수신,194324,11.06
3,운동기록,131269,7.47
4,마음챙김,130344,7.42
5,식단기록,101366,5.77
6,챌린지참여,96829,5.51
7,챌린지_탐색,78101,4.44
8,NaN,26456,1.51
9,알림오픈,21219,1.21



Event_Log의 알림_유형 값 확인
전체 행 수: 1757262
알림_유형 고유값 개수(결측 포함): 4

[범주별 요약]


,알림_유형,개수,비율(%)
0,NaN,1538380,87.54
1,리마인드,85830,4.88
2,광고성,78262,4.45
3,챌린지_알림,54790,3.12


In [121]:
# 8. 데이터 간 키 연결 상태 확인
check_user_id_relation( user_profile, event_log)


User_Profile - Event_Log User_ID 연결 확인


,항목,값
0,프로필 User_ID 고유 수,12500
1,이벤트 User_ID 고유 수,12453
2,양쪽 모두 존재하는 User_ID 수,12453
3,프로필에만 있는 User_ID 수,47
4,이벤트에만 있는 User_ID 수,0



[프로필에는 있지만 이벤트 로그에는 없는 User_ID 샘플]


,User_ID
0,U0004830
1,U0011201
2,U0011432
3,U0002436
4,U0002415
5,U0005343
6,U0003870
7,U0002163
8,U0010532
9,U0008366


# 5. 수치적 데이터 탐색 보고서
본격적인 전처리에 앞서, 원천 데이터의 구조와 품질 상태를 먼저 확인.
전처리 과정에서 어떤 값을 제거, 보정, 유지할지 판단하기 위해 다음 항목들을 중심으로 데이터를 점검

## 1. 기본 구조 확인
- `User_Profile`은 사용자 단위 데이터이고, **총 12,500행 × 6열**
- `Event_Log`는 사용자 행동 로그 단위 데이터이고, **총 1,757,262행 × 5열**

---

## 2. 결측값 상태 확인
각 컬럼별 결측치 개수와 결측치 비율을 확인

### User_Profile
User_ID와 가입일자엔 결측치 없음

- `알림수신동의_변경일자`: **10,524건 결측, 84.19%**
- `가입경로`: **137건 결측, 1.10%**
- `기기`: **121건 결측, 0.97%**
- `알림수신동의여부`: **116건 결측, 0.93%**

`알림수신동의_변경일자`는 결측 비율이 높지만, 이는 알림 수신 동의 상태가 변경되지 않은 사용자를 의미할 가능성이 있다.   
따라서 단순 제거 대상이라기보다는 **변경 이력 없음**으로 해석 가능한지 확인이 필요하다.

`가입경로`, `기기`, `알림수신동의여부`의 결측 비율은 약 1% 수준으로 낮다.   
다만, 사용자 세그먼트 분석에 사용될 수 있으므로 전처리 기준을 명확히 정해야 한다.

### Event_Log
User_ID와 Event_Time에 결측치 없음

- `알림_유형`: **1,538,380건 결측, 87.54%**
- `Session_ID`: **241,502건 결측, 13.74%**
- `Event_Type`: **26,456건 결측, 1.51%**

`알림_유형`은 알림 관련 이벤트에만 값이 존재하는 컬럼이므로, 전체 결측 비율만 보고 문제로 판단하기 어렵다.  
`Session_ID` 역시 알림 이벤트는 앱 외부에서 발생하므로 결측이 정상일 수 있다.

`Event_Type` 결측은 어떤 행동인지 알 수 없는 로그이므로, 분석 전 **제거할지, 별도 Unknown 값으로 유지할지** 처리 기준을 정해야 한다.

---

## 3. 중복 데이터 확인
중복 여부는 세 가지 관점에서 확인

### 3-1. 완전 중복 행 확인
모든 컬럼 값이 완전히 동일한 행이 존재하는지 확인
- User_Profile과 Event_Log 모두 **완전 중복 행은 0건**

단순히 동일한 행이 중복 적재된 문제는 현재 단계에서 확인되지 않았다

### 3-2. 기준 키 중복 확인
데이터의 기준 키로 사용할 수 있는 컬럼의 중복 여부를 확인
- User_Profile에서는 User_ID를 기준으로 중복 여부를 확인
    - User_ID는 **12,500개 모두 고유값**이며, 중복 User_ID는 **0건**이었다.
- Event_Log에서는 단일 User_ID가 여러 이벤트를 가질 수 있으므로, User_ID 단독 중복은 문제가 아니다.

### 3-3. ID는 다르지만 값 자체가 동일한 경우
- User_Profile에서는 User_ID를 제외하고 확인한 결과, **9,121건의 중복 행**이 확인   
로 다른 사용자가 동일한 프로필 조합을 갖는 것은 자연스러울 수 있으므로, 이 결과를 즉시 제거 대상으로 보기는 어렵다.


- Event_Log에서는 User_ID와 Session_ID를 제외하고 확인한 결과, **87,794건의 중복 행**이 확인    
이는 서로 다른 사용자가 같은 시간대에 같은 이벤트를 발생시켰기 때문일 수 있다.  
다만 로그 적재 과정에서 ID만 다르게 기록된 중복 가능성도 있으므로, 특정 시간대나 특정 이벤트에 과도하게 몰려 있는지 추가 확인이 필요하다.

---

## 4. 날짜 변환 상태 확인
날짜 관련 컬럼이 정상적으로 `datetime` 타입으로 변환되는지 확인 
확인 대상은 다음과 같다.

`가입일자`는 원본 결측치와 변환 실패가 모두 **0건**이었다.  
날짜 범위는 **2025-01-01부터 2025-05-26까지**로 확인되었다.

`알림수신동의_변경일자`는 원본 결측치가 **10,524건**이었지만, 값이 존재하는 경우 날짜 변환 실패는 **0건**이었다.    
날짜 범위는 **2025-01-21부터 2025-06-30까지**로 확인되었다.

`Event_Time`은 원본 결측치와 변환 실패가 모두 **0건**이었다.    
이벤트 로그의 시간 범위는 **2025-01-01 07:00:07부터 2025-06-30 22:59:51까지**로 확인되었다.

따라서 주요 날짜 컬럼은 모두 정상적으로 날짜 변환이 가능하다.

---
## 5. 기술통계 및 이상치 확인

### 5-1. 사용자별 이벤트 수

사용자별 이벤트 수는 총 **12,453명**을 기준으로 확인하였다.

사용자 1명당 평균 이벤트 수는 **141.11건**이었고, 중앙값은 **51건**이었다.
Q1은 **20건**, Q3는 **219건**, 최댓값은 **697건**으로 확인되었다.

왜도는 **1.33**으로 양의 왜도를 보였다.
즉, 대부분의 사용자는 비교적 적은 이벤트를 남기지만, 일부 사용자는 매우 많은 이벤트를 남기는 구조로 해석할 수 있다.

IQR 기준 이상치 후보는 **681명, 5.47%**로 확인되었다.
사용자별 이벤트 수의 이상치 기준 상한값은 **517.5건**이며, 이를 초과하는 사용자가 이상치 후보로 분류되었다.

다만 이벤트 수가 많은 사용자는 실제 충성 사용자일 수 있으므로, 단순 제거하기보다는 **헤비 유저 또는 고활동 사용자 세그먼트**로 분리해 해석하는 것이 적절하다.

### 5-2. 세션별 이벤트 수

세션별 이벤트 수는 총 **736,281개 세션**을 기준으로 확인하였다.

세션당 평균 이벤트 수는 **2.06건**이었고, 중앙값은 **2건**이었다.
Q1은 **1건**, Q3는 **3건**, 최댓값은 **8건**으로 확인되었다.

세션별 이벤트 수는 전반적으로 낮은 편이며, 하나의 세션 안에서 많은 이벤트가 연속적으로 발생하는 구조는 강하지 않은 것으로 보인다.

IQR 기준 이상치 후보는 **183건, 0.02%**로 매우 적었다.
이상치 기준 상한값은 **6건**이며, 이를 초과하는 세션이 이상치 후보로 분류되었다.

세션별 이상치 후보 비율이 매우 낮기 때문에, 현재 단계에서는 세션 단위의 심각한 이상 패턴은 크게 확인되지 않았다.

### 5-3. 일별 이벤트 수

일별 이벤트 수는 총 **181일**을 기준으로 확인하였다.

일평균 이벤트 수는 **9,708.63건**이었고, 중앙값은 **11,358건**이었다.
최솟값은 **665건**, 최댓값은 **16,495건**으로 확인되었다.

IQR 기준 이상치 후보는 **0건**으로 확인되었다.
즉, IQR 기준에서는 일별 이벤트 수가 통계적으로 극단적인 날짜는 확인되지 않았다.

다만 과제 안내상 **2025년 3월 10일부터 3월 14일까지 로그 수집 장애 기간**이 존재하므로, IQR 이상치 여부와 별개로 해당 기간의 이벤트 수 변화는 별도 확인이 필요하다.

---

## 6. 범주형 값 분포 확인

범주형 컬럼에 어떤 값이 존재하는지 확인하였다.
확인 대상은 `가입경로`, `기기`, `알림수신동의여부`, `Event_Type`, `알림_유형`이다.

### User_Profile 범주형 컬럼

`가입경로`는 `퍼포먼스광고`가 **6,852건, 54.82%**, `오가닉`이 **5,511건, 44.09%**로 확인되었다.
결측치는 **137건, 1.10%**였다.

`기기`는 `iOS`가 **7,175건, 57.40%**, `Android`가 **5,204건, 41.63%**로 확인되었다.
결측치는 **121건, 0.97%**였다.

`알림수신동의여부`는 `True`가 **7,984건, 63.87%**, `False`가 **4,400건, 35.20%**로 확인되었다.
결측치는 **116건, 0.93%**였다.

전반적으로 `User_Profile`의 범주형 컬럼은 예상 가능한 값으로 구성되어 있으며, 결측 비율도 `알림수신동의_변경일자`를 제외하면 약 1% 수준이다.

### Event_Log 범주형 컬럼

`Event_Type`은 결측을 포함해 총 **11개 값**으로 확인되었다.

주요 이벤트 분포는 다음과 같다.

* `앱실행`: **728,657건, 41.47%**
* `수면기록`: **242,978건, 13.83%**
* `알림수신`: **194,324건, 11.06%**
* `운동기록`: **131,269건, 7.47%**
* `마음챙김`: **130,344건, 7.42%**
* `식단기록`: **101,366건, 5.77%**
* `챌린지참여`: **96,829건, 5.51%**
* `챌린지_탐색`: **78,101건, 4.44%**
* `Event_Type` 결측: **26,456건, 1.51%**
* `알림오픈`: **21,219건, 1.21%**
* `온보딩_완료`: **5,719건, 0.33%**

`Event_Type`에서는 `앱실행`이 가장 큰 비중을 차지했다.
또한 `Event_Type` 결측이 **26,456건** 존재하므로, 해당 로그를 제거할지 별도 범주로 둘지 판단해야 한다.

`알림_유형`은 결측을 포함해 총 **4개 값**으로 확인되었다.

* 결측: **1,538,380건, 87.54%**
* `리마인드`: **85,830건, 4.88%**
* `광고성`: **78,262건, 4.45%**
* `챌린지_알림`: **54,790건, 3.12%**

`알림_유형`의 결측 비율은 높지만, 해당 컬럼은 알림 이벤트에만 값이 존재하는 구조이므로 전체 결측률만으로 문제라고 판단하기 어렵다.
이후 알림 이벤트인 `알림수신`, `알림오픈`과 `알림_유형`의 관계를 별도로 확인할 필요가 있다.

---

## 7. 키 연결 상태 확인
`User_Profile`과 `Event_Log`는 `User_ID`를 기준으로 연결되는 데이터이기 때문에  
따라서 두 데이터 간 키 연결 상태를 확인

- `User_Profile`의 고유 `User_ID` 수는 **12,500명**
- `Event_Log`에 등장한 고유 `User_ID` 수는 **12,453명**
- 두 데이터에 모두 존재하는 사용자는 **12,453명**

다만 `User_Profile`에는 존재하지만 `Event_Log`에는 없는 사용자가 47명 존재했다. 


# 전처리 전 판단이 필요한 항목
첫째, `Event_Type` 결측치 **26,456건**의 처리 기준을 정해야 한다.
행동 유형을 알 수 없는 로그이므로, 분석 목적에 따라 제거하거나 별도 `Unknown` 범주로 유지할 수 있다.

둘째, `Session_ID`와 `알림_유형`의 결측은 컬럼 의미에 따라 해석해야 한다.
알림 이벤트는 앱 외부에서 발생하므로 `Session_ID`가 없는 것이 정상일 수 있고, `알림_유형`은 알림 이벤트에만 값이 존재할 가능성이 높다.

셋째, ID를 제외한 값 기준 중복은 바로 제거하지 않는다.
`User_Profile`의 **9,121건**, `Event_Log`의 **87,794건**은 서로 다른 사용자 또는 세션에서 같은 값 조합이 발생한 것일 수 있으므로, 실제 중복 적재 여부를 추가 확인한 뒤 처리해야 한다.

넷째, 사용자별 이벤트 수 이상치 후보 **681명**은 단순 이상치라기보다 고활동 사용자일 가능성이 있다.
따라서 제거보다는 별도 세그먼트로 구분해 리텐션 차이를 비교하는 방향이 적절하다.

다섯째, 프로필에는 있지만 이벤트 로그에는 없는 **47명**의 처리 기준을 정해야 한다.
이들을 가입 후 미활동 사용자로 볼 경우 리텐션 분모에 포함될 수 있고, 로그 누락으로 볼 경우 별도 제외 기준이 필요하다.

마지막으로, 30일 리텐션 분석을 위해 **D30 관측 가능 사용자**를 별도로 구분해야 한다.
가입일자가 늦은 사용자는 분석 기간 내 D30을 관측할 수 없으므로, 리텐션 계산 시 분모에서 제외하거나 별도 코호트로 분리해야 한다.

# 6. 기초 전처리

In [127]:
# ============================================================
# 1. 기초 전처리용 데이터 복사
# ============================================================
user_profile_base = user_profile.copy()
event_log_base = event_log.copy()

In [136]:
# ============================================================
# 2. 공백 제거
# - 컬럼명과 문자열 값의 앞뒤 공백 제거
# - 문자열이 아닌 값은 그대로 유지
# ============================================================

# 컬럼명 앞뒤 공백 제거
user_profile_base.columns = [
    col.strip() if isinstance(col, str) else col
    for col in user_profile_base.columns
]

event_log_base.columns = [
    col.strip() if isinstance(col, str) else col
    for col in event_log_base.columns
]

In [137]:
# ============================================================
# 3. User_ID 타입 통일
# - 두 데이터를 User_ID로 연결하기 위해 문자열 타입으로 통일
# ============================================================
user_profile_base["User_ID"] = user_profile_base["User_ID"].astype(str)
event_log_base["User_ID"] = event_log_base["User_ID"].astype(str)

In [138]:
# ============================================================
# 4. 날짜 컬럼 datetime 변환
# - 변환 실패값은 NaT로 남김
# - 아직 행 제거는 하지 않음
# ============================================================
user_profile_base["가입일자"] = pd.to_datetime(
    user_profile_base["가입일자"],
    errors="coerce"
)

user_profile_base["알림수신동의_변경일자"] = pd.to_datetime(
    user_profile_base["알림수신동의_변경일자"],
    errors="coerce"
)

event_log_base["Event_Time"] = pd.to_datetime(
    event_log_base["Event_Time"],
    errors="coerce"
)

In [139]:
# ============================================================
# 5. 날짜 파생 컬럼 생성 (임시)
# - 이후 코호트, 리텐션, 일별 이벤트 수 분석에 사용
# - 아직 Day 0, Day 30 기준을 정하지 않음
# ============================================================

user_profile_base["가입일"] = user_profile_base["가입일자"].dt.date
user_profile_base["가입월"] = user_profile_base["가입일자"].dt.to_period("M").astype(str)

event_log_base["Event_Date"] = event_log_base["Event_Time"].dt.date
event_log_base["Event_Month"] = event_log_base["Event_Time"].dt.to_period("M").astype(str)
event_log_base["Event_Hour"] = event_log_base["Event_Time"].dt.hour

In [140]:
# ============================================================
# 9. 기본 전처리 결과 확인
# ============================================================

print("user_profile 원본 shape :", user_profile.shape)
print("user_profile_base shape :", user_profile_base.shape)

print("event_log 원본 shape    :", event_log.shape)
print("event_log_base shape    :", event_log_base.shape)

display(user_profile_base.head())
display(event_log_base.head())

user_profile 원본 shape : (12500, 6)
user_profile_base shape : (12500, 8)
event_log 원본 shape    : (1757262, 5)
event_log_base shape    : (1757262, 8)


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자,가입일,가입월
0,U0000001,2025-01-25,오가닉,iOS,True,NaT,2025-01-25,2025-01
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24,2025-05-06,2025-05
2,U0000003,2025-05-14,오가닉,iOS,False,NaT,2025-05-14,2025-05
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaT,2025-02-23,2025-02
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaT,2025-02-18,2025-02


,User_ID,Event_Time,Event_Type,Session_ID,알림_유형,Event_Date,Event_Month,Event_Hour
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,NaN,2025-01-25,2025-01,7
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,NaN,2025-01-25,2025-01,7
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,NaN,2025-01-25,2025-01,7
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,NaN,2025-01-25,2025-01,7
4,U0000001,2025-01-25 20:30:00,알림수신,NaN,광고성,2025-01-25,2025-01,20
